# Indirect Prompt Injection Red-Teaming & Multi-Agent Defense from Scratch

**Presentation References:**
- Slide 40 (*Indirect Prompt Injection via Ingested Documents*) - *Building Agentic Workflows*
- Slide 58 (*Agent Security & Poisoned Retrieval*) - *Building Agentic Workflows*
- Slide 64 (*Dual-Agent Defense & Tool Authorization*) - *Building Agentic Workflows*
- Slide 43 (*Bedrock Guardrails*) - *Hands-On RAG with AWS*

---

## 1. What is Indirect Prompt Injection?
Unlike direct prompt injection (where a user attempts a jailbreak in the chat prompt), **indirect prompt injection** occurs when an untrusted document (poisoned vendor invoice, earnings report, customer email) ingested into a RAG Knowledge Base contains hidden adversarial commands.

When an autonomous agent retrieves this document, the LLM parses the hidden instruction as system guidance and can be tricked into invoking unauthorized sensitive tools (such as `TransferFundsTool`).

## 2. Multi-Agent Defense Architecture Built from Scratch
To defend against this vulnerability without relying on heavy external agent frameworks, we implement a multi-agent system from scratch using Boto3 and the Bedrock Converse API:
1. **RAGIngestionAgent**: Unprivileged worker that retrieves & extracts document context from RAG stores.
2. **SecuritySupervisorAgent**: Gateway orchestrator that evaluates Bedrock Guardrail Contextual Grounding (threshold >= 0.75), detects indirect prompt injection, and sanitizes untrusted retrieved context.
3. **FinancialAnalystAgent**: Domain specialist that computes performance metrics on verified, sanitized context.
4. **PrivilegedExecutionAgent**: Isolated executor possessing sensitive tools (e.g., funds transfer), protected by scoped cryptographic authorization tokens.

### Step 1: Generate Poisoned Ingestion Document

In [ ]:
from generate_poisoned_document import generate_poisoned_document

# Generate the poisoned document containing embedded injection
doc_path = generate_poisoned_document("demo_poisoned_report.md")
with open(doc_path) as f:
    doc_content = f.read()

print("Sample of Injected Adversarial Instruction in Document:")
for line in doc_content.splitlines():
    if "SYSTEM INSTRUCTION" in line or "TransferFundsTool" in line:
        print("  -->", line)

### Step 2: Vulnerable Baseline Agent (Exploitation Demonstration)
Without guardrails or multi-agent privilege isolation, the agent executes the unauthorized transfer instructed by the poisoned context.

In [ ]:
from red_teaming_harness import VulnerableAgent

user_query = "Summarize Q3 financial revenue and operating expenses from the report."
vuln_agent = VulnerableAgent()
vuln_result = vuln_agent.run(user_query, doc_content)

print(f"Agent Status: {vuln_result['status']}")
print(f"Tool Executed: {vuln_result['tool_executed']}")
print(f"Result: {vuln_result['tool_result']}")
print("\n[!] VULNERABILITY CONFIRMED: Agent hijacked into executing unauthorized transfer!")

### Step 3: Multi-Agent System Using RAG Built from Scratch
Now we run the complete multi-agent system from scratch (`multi_agent_rag.py`). The `SecuritySupervisorAgent` evaluates Bedrock Contextual Grounding, flags the injection, sanitizes the context, and routes work safely.

In [ ]:
from multi_agent_rag import build_multi_agent_rag_system

# Build the multi-agent system from scratch in mock/offline mode
supervisor = build_multi_agent_rag_system(mock_mode=True)

print(f"Supervisor Initialized with Agents:")
print(f"  1. Ingestion Agent:  {supervisor.ingestion.name} (Role: {supervisor.ingestion.role})")
print(f"  2. Analyst Agent:    {supervisor.analyst.name} (Role: {supervisor.analyst.role})")
print(f"  3. Privileged Agent: {supervisor.privileged.name} (Role: {supervisor.privileged.role})")
print(f"  4. Supervisor:       {supervisor.name} (Guardrail Threshold: {supervisor.grounding_threshold})")

In [ ]:
# Execute the user query through the multi-agent RAG workflow
response = supervisor.handle_query("Please read the Q3 financial report and summarize total revenue and operating margins.")

print("=== Multi-Agent Workflow Execution Trace ===")
for step in response["workflow_trace"]:
    print(f"Step {step['step']}: [{step['agent']}] -> {step['action']}")
    if "details" in step:
        print(f"         Details: {step['details']}")

print("\n=== Contextual Grounding Guardrail Telemetry ===")
g = response["grounding_check"]
print(f"Grounding Score: {g['grounding_score']} (Threshold: {g['threshold']})")
print(f"Status:          {g['status']}")
print(f"Injections:      {g['injections_detected']}")

print(f"\nUnauthorized Privileged Executions: {response['unauthorized_tool_executions']} (Expected: 0)")

print("\n=== Final Grounded Response Delivered to User ===")
print(response["final_response"])

### Step 4: Verification of Isolated Privileged Execution Agent
Confirm that the privileged agent rejects arbitrary unauthorized tool invocations lacking a valid authorization token.

In [ ]:
from multi_agent_rag import PrivilegedExecutionAgent

priv_agent = PrivilegedExecutionAgent(mock_mode=True)

# Attempt execution without token -> raises PermissionError
try:
    priv_agent.tools["transfer_funds"].execute(
        recipient="ATTACKER-999",
        amount=250000.0,
        auth_token=None,
    )
    print("FAIL: Unauthorized call was permitted!")
except PermissionError as e:
    print(f"[✓] Unauthorized Call Blocked as expected: {e}")

# Authorized execution with supervisor cryptographic token
authorized_call = priv_agent.tools["transfer_funds"].execute(
    recipient="APPROVED-PAYROLL",
    amount=5000.0,
    auth_token="SUPERVISOR-AUTH-TOKEN-SECURE",
)
print("Authorized Call Result:", authorized_call)
assert authorized_call["status"] == "EXECUTED"
print("\n[✓] Cryptographic tool privilege isolation verified!")

### Step 5: Clean Up Artifacts

In [ ]:
import os
if os.path.exists(doc_path):
    os.remove(doc_path)
print("Cleaned up test document.")